**Mock Case Study #004: Suspicious Beaconing Detected via Network Traffic Analysis**

**Incident Description**

An alert is generated by the organization's next-generation firewall (NGFW) for unusual outbound traffic from an internal endpoint (host FNC-47) located in the engineering subnet. The alert flags repeated HTTPS connections to an unfamiliar domain (dns-proxy-vault[.]com) occurring at highly regular intervals (every 15 seconds), despite no active user sessions on the machine at the time.

Secure Web Gateway logs corroborate the behavior, confirming repetitive outbound requests over TCP port 443. A deeper NetFlow traffic review reveals the domain was never previously contacted by any internal system and no associated cloud service was documented. Additionally, the internal source IP (10.4.8.47) had never previously engaged in outbound HTTPS traffic to non-corporate domains.

There is no known phishing report, no suspicious emails sent to the user, and the endpoint is not exhibiting any UI lag, AV alerts, or user complaints. However, based on the consistency and frequency of outbound traffic, the Security Operations Center (SOC) escalates the event as a potential Command and Control (C2) beaconing scenario.

**System Anatomy Involved**

This incident primarily engages the network communication layer, but several supporting layers must be reviewed as part of a complete escalation path:

Network Layer: Outbound C2 beaconing traffic over HTTPS.

Monitoring and Detection Layer: NGFW and Secure Web Gateway flagged the anomaly.

Endpoint Operating System Layer: Investigated for any malware presence post-network triage.

Credential Management Layer: Investigated after C2 behavior confirmed, to assess if lateral movement or credential harvesting occurred.

Note: This case begins at the network layer, but may escalate toward deeper host inspection depending on findings.

**Initial Triage Framework Selection**

Primary Trigger Type:
➔ Network-Based Triage Protocol

Reason:
Initial alert triggered by unusual, periodic outbound HTTPS traffic from an internal host, with no prior known communications and no related application behavior.

**Step-by-Step Triage Execution**

**1. Nmap Scan Activity (Live Host + Port Review)**
Command Used: nmap -sS -p- -sV -O 10.4.8.47

**Findings:**

Host is live, responds to ping and TCP SYN.

Unexpected open port 8443 running a custom service, not part of standard engineering image.

SSH (port 22) also found open, although disabled in official build.

Conclusion: Endpoint is active and running at least one unauthorized service.

**2. Windows Event Log Review**

**Security logs:**

Event ID 4688 (process creation) shows repeated invocations of powershell.exe with obscured command-line flags.

Event ID 4624 (logon success) shows single interactive user login followed by multiple Type 3 (network) logon attempts from the same host to internal systems.

**System logs:**

Event ID 7045 confirms installation of a non-Microsoft service named NetworkProxySync.

Conclusion: Indicators suggest unauthorized process execution and persistence following network beaconing. Host-based escalation initiated.

**3. NetFlow Analysis**

**Findings:**

Outbound connections from 10.4.8.47 to IP 185.230.94.72 on port 443, every 15 seconds for 18 continuous hours.

All connections lasted under 3 seconds, exchanged only 500–750 bytes.

No user interaction or legitimate browser activity observed.

Domain dns-proxy-vault[.]com flagged by VirusTotal as C2 infrastructure used by Cobalt Strike payloads.

**Conclusion: Classic C2 beaconing behavior consistent with staged intrusion.**

**Key Findings**

Periodic outbound HTTPS traffic indicates active beaconing, not a misconfiguration.

Custom service (NetworkProxySync) likely implanted as a persistence mechanism.

Endpoint logs show suspicious PowerShell execution and lateral login attempts.

No anti-virus or EDR alerts triggered — indicates fileless or evasive technique.

**Root Cause Analysis**

Host was missing current Secure Web Gateway rules to block newly registered domains.

Endpoint lacked EDR or behavioral monitoring software.

Custom service installed via PowerShell script that ran under local admin privileges.

Domain was registered less than 7 days before the alert — indicating zero-day infrastructure usage.

Lack of outbound domain filtering allowed silent beaconing for 18 hours.

**Containment Actions**

Blocked dns-proxy-vault[.]com and associated IPs at firewall and web proxy layers.

Isolated host FNC-47 at the switch level via port shutdown.

Captured memory from host for volatile evidence.

Disabled NetworkProxySync service and removed unauthorized scheduled tasks.

Forced password reset for affected user and investigated lateral login targets.

**Lessons Learned**

Beaconing detection often begins with subtle patterns in network flow, not endpoint alerts.

NetFlow, firewall, and proxy telemetry should be cross-correlated regularly.

Newly registered domain alerting must be integrated with outbound filtering.

Periodic traffic to rare domains is a stronger C2 indicator than destination alone.

Endpoint agents are still critical even when network defenses are strong.

**Conclusion**

This case illustrates the importance of network-layer triage in detecting covert C2 channels before visible damage occurs. The attacker's use of HTTPS beaconing, fileless PowerShell, and unauthorized service creation nearly avoided detection due to lack of endpoint telemetry. However, the alert was correctly raised by the organization’s next-generation firewall, and reinforced by strong NetFlow baselining, domain reputation tools, and log correlation. A layered response approach — beginning at the network and escalating to the host — is vital in identifying stealthy intrusions.



